# Field Shift — Monsoon Onset ML Outlook (`onset-ml-v3` & Phase 5 Experiments)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zul-Qarnain/field-shift/blob/main/notebooks/field_shift_onset_ml_colab.ipynb)

This notebook runs the **Field Shift** Monsoon Onset ML pipeline (`scripts/build_onset_ml.py`) and the **Phase 5 ML experiments** (`EXPERIMENT_LOG.md`) in Google Colab.

### Key Properties
- **100% Public NASA / JAXA Data (Zero Credentials Required):** Streams directly from NASA POWER's public v10 Zarr stores on AWS Open Data (`imerg_daily`, `merra2_daily`, `merra2_monthly`).
- **Strict Leave-One-Year-Out (LOYO) Validation:** Every year (1998–2025, 28 folds × 64 districts = 1,792 district-years) is held out across all districts simultaneously. Fold-local climatologies exclude both the held-out test year and each training row's own year.
- **Split-Conformal 80% Intervals:** Calibrated on inner year-grouped out-of-fold residuals.

In [ ]:
# 1. Clone the repository (if running on Google Colab) and install dependencies
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("/content/field-shift").exists():
        !git clone https://github.com/Zul-Qarnain/field-shift.git /content/field-shift
    os.chdir("/content/field-shift")
    !pip install -q -r requirements.txt scikit-learn matplotlib seaborn
else:
    # Local execution: ensure repo root is on sys.path
    repo_root = Path.cwd().resolve()
    if repo_root.name == "notebooks":
        os.chdir(repo_root.parent)

sys.path.insert(0, str(Path.cwd()))
print("Working directory:", Path.cwd())

## 1. Schema & NASA Data Ingestion

We load the 64 Bangladesh districts and 6 demo farm coordinates, then fetch:
1. **NASA/JAXA GPM IMERG V07 (`0.1°` daily precipitation, 1998–2026)** for `v2` monsoon onset labels and 1 Mar – 20 Apr rainfall accumulation.
2. **NASA POWER v10 (`MERRA-2` daily meteorology, 1981–2026)** for pre-monsoon mean/max temperature (`T2M`, `T2M_MAX`), relative humidity (`RH2M`), and end-of-window root-zone soil wetness (`GWETROOT`).
3. **NASA MERRA-2 Monthly Sea-Surface Skin Temperature (`TS`)** for Niño 3.4 (`DJF`, `FMA`) and the Indian Ocean Dipole Mode Index (`DMI`, `Feb–Mar`).

> **Tip:** Set `SMOKE_LIMIT = 0` below for the full 64-district run (~8–12 minutes on first uncached S3 download, ~25 seconds on cached reruns), or `SMOKE_LIMIT = 8` for a fast 1-minute test.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scripts import build_onset_ml as B
from src.acquire import imerg, indices
from src.compute.ml import dataset as D, features as F, onset_ml as M, outlook as O

# Set SMOKE_LIMIT = 0 for all 64 districts + 6 farms, or e.g. 8 for a quick smoke test
SMOKE_LIMIT = 0

t0 = time.time()
oni, dmi = indices.nino34(), indices.dmi()
points = B.load_points()
if SMOKE_LIMIT > 0:
    points = [p for p in points if p.kind == "farm"] + [p for p in points if p.kind == "district"][:SMOKE_LIMIT]

power_frames = {}
imerg_series = {}
cells = {}
stamps = []

for idx, point in enumerate(points, 1):
    frame = B.read_power(point)
    power_frames[point.key] = frame
    stamps.append(str(frame.attrs.get("retrieved_utc", "unknown")))
    imerg_series[point.key] = B.read_imerg(point)
    cells[point.key] = imerg.cell_of(point.lat, point.lon)
    if idx % 10 == 0 or idx == len(points):
        print(f"Loaded {idx}/{len(points)} points ({time.time() - t0:.1f}s)")

variant_imerg = D.build_variant("imerg", points, power_frames, imerg_series, B.OUTLOOK_YEAR)
variant_imerg.cells = cells
years = D.year_pool(variant_imerg)
districts = [p.district() for p in variant_imerg.points if p.kind == "district"]
print(f"Ready: {len(districts)} districts across {len(years)} seasons ({years[0]}–{years[-1]})")

### Analysis of the Ingested Dataset
- Each district has a yearly time series of `v2` monsoon onset Day-of-Year (`onset_doy`) bounded to the diagnostic window `DOY 121–243` (May 1 – Aug 31).
- Every dynamic weather predictor is expressed as a **fold-local anomaly** relative to that district's training-year climatology (`FEATURE_NAMES`), preventing spatial differences in mean rainfall or temperature from leaking into temporal anomaly trees.

## 2. Exploratory Data Analysis & Target Distribution

Before training models, we inspect:
1. The distribution of monsoon onset DOY across districts and years (checking for outliers or boundary censoring at DOY 121).
2. The national mean onset trajectory over time (`1998–2025`) with inter-district spread.
3. Relationships between the onset anomaly target (`onset_doy - base_onset`) and the 11 pre-April-25 features.

In [ ]:
eda_df = B.arm_rows(variant_imerg, years, oni, dmi)
train_eda = eda_df[eda_df["year"].isin(years) & eda_df["district"].isin(variant_imerg.trainable)].dropna(subset=["onset_doy", "base_onset"]).copy()
train_eda["onset_anom"] = train_eda["onset_doy"] - train_eda["base_onset"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Plot 1: Distribution of Onset DOY and Anomaly
axes[0].hist(train_eda["onset_doy"], bins=24, color="#2d6a4f", edgecolor="white", alpha=0.85)
axes[0].axvline(train_eda["onset_doy"].mean(), color="#d97706", linestyle="--", linewidth=2, label=f"Mean DOY ({train_eda['onset_doy'].mean():.1f})")
axes[0].set_title("Distribution of v2 Monsoon Onset (DOY)")
axes[0].set_xlabel("Day of Year (DOY 121 = May 1)")
axes[0].set_ylabel("District-Years")
axes[0].legend()

# Plot 2: Chronological National Mean Onset DOY (1998-2025)
yearly_stats = train_eda.groupby("year")["onset_doy"].agg(["mean", "std", "median"])
axes[1].plot(yearly_stats.index, yearly_stats["mean"], marker="o", color="#1d3557", label="National Mean DOY")
axes[1].fill_between(
    yearly_stats.index,
    yearly_stats["mean"] - yearly_stats["std"],
    yearly_stats["mean"] + yearly_stats["std"],
    color="#a8dadc",
    alpha=0.4,
    label="±1 SD across districts",
)
axes[1].set_title("Monsoon Onset DOY Over Time (1998–2025)")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Onset DOY")
axes[1].legend()

# Plot 3: Feature Correlations with Onset Anomaly
corrs = train_eda[F.FEATURE_NAMES].apply(lambda col: col.corr(train_eda["onset_anom"])).sort_values()
corrs.plot(kind="barh", ax=axes[2], color=["#e63946" if v < 0 else "#457b9d" for v in corrs])
axes[2].axvline(0, color="black", linewidth=0.8)
axes[2].set_title("Linear Correlation with Onset Anomaly (d)")
axes[2].set_xlabel("Pearson r")

plt.tight_layout()
plt.show()

spread = B.label_spread(variant_imerg, years)
print(f"Between-district SD: {spread['between_district_sd_days']} days | Within-district year-to-year SD: {spread['within_district_sd_days']} days")
print("Top correlations with onset anomaly:\n", corrs.round(4))

### Analysis of EDA Findings
- **Within-district vs. Between-district Variance:** Year-to-year variability within a district (~14 days SD) is more than double the spatial spread between district climatologies (~6.4 days SD). Predicting the **anomaly** (`onset_doy - base_onset`) isolates the inter-annual signal.
- **Pre-monsoon Teleconnection & Local Signal:** Pre-April-25 linear correlations with onset timing are modest ($|r| < 0.15$), explaining why an unregularized tree model can easily overfit year-level predictors (`nino34_djf`, `nino34_fma`, `dmi_feb_mar`) when there are only 28 distinct monsoon years.

## 3. Baseline Reproduction (`onset-ml-v3` Leave-One-Year-Out Backtest)

We now run the exact `onset-ml-v3` Leave-One-Year-Out (LOYO) backtest across all 28 folds (`1998–2025`) to verify our reproduction of `src/compute/ml/metrics.json`.

In [ ]:
samples = M.build_samples(districts, variant_imerg.scalars, variant_imerg.labels, oni, dmi, years)
folds = M.backtest_rows(samples, M.leave_one_year_split(years))
baseline_report = M.run_backtest(folds, rounds=M.NUM_BOOST_ROUND)
bm = baseline_report["metrics"]

print("=== Baseline onset-ml-v3 (LOYO Backtest) ===")
print(f"District-years : {bm['n_district_years']}")
print(f"Model MAE      : {bm['mae_model']:.2f} d  (Climatology MAE: {bm['mae_clim']:.2f} d | Skill: {bm['skill']:+.3f})")
print(f"vs 2-3yr mean  : {bm['mae_clim2']:.2f} d  (Skill: {bm['skill_vs_clim2']:+.3f})")
print(f"vs Last season : {bm['mae_lag1']:.2f} d  (Skill: {bm['skill_vs_lag1']:+.3f})")
print(f"RMSE           : {bm['rmse']:.2f} d")
print(f"80% Conformal  : {bm['coverage']*100:.1f}% coverage")
print(f"Status         : {bm['status']}")

### Analysis of Shipped `onset-ml-v3` ($L_1$ Local-Feature Anomaly Model)
- By training with **`objective="regression_l1"` (MAE loss)** and restricting splits via `_LocalFeatureBooster` to the **8 district-varying local features** (so year-constant ocean indices `nino34_djf`, `nino34_fma`, `dmi_feb_mar` cannot isolate and memorize individual training years), `onset-ml-v3` achieves **`mae_model = 10.63 d` vs `mae_clim = 10.73 d` (`skill = +0.009`)**, beating district climatology in **17 of 28 held-out years (60.7%)** (`status = "shipped"`).
- It also beats the 2–3-season mean (`12.98 d`, `skill_vs_clim2 = +0.181`) and prior-season persistence (`14.80 d`, `skill_vs_lag1 = +0.281`), while maintaining calibrated 80% split-conformal coverage (**79.7%**).

## 4. Phase 5 ML Experiments: Model & Hyperparameter Comparison in LOYO CV

Why did the 150-round LightGBM booster lose to climatology before shrinkage? With only **28 monsoon years**, boosting for 150 rounds memorizes year-level noise. Here we compare four modeling strategies across the exact same 28 Leave-One-Year-Out folds:
1. **Baseline LightGBM (150 rounds, 1-SE shrinkage)** — current shipped configuration.
2. **Ultra-Shallow Early-Stopped LightGBM (25 rounds, `max_depth=2`, `num_leaves=3`, `lambda_l2=50`)** — heavily damped tree ensemble designed for 28-year macro regimes.
3. **Standardized Ridge Regression ($L_2$ Linear Anomaly Model, $\alpha=500$)** — linear combination of standardized pre-monsoon anomalies.
4. **Huber Robust Regression ($L_1/L_2$ Hybrid)** — directly optimizes MAE-like robust loss on standardized anomalies, matching the MAE evaluation metric.

In [ ]:
from sklearn.linear_model import Ridge, HuberRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

def evaluate_custom_loyo(folds, fit_predict_fn, name: str, shrink_factor: float = 1.0):
    """Evaluate any anomaly model across the 28 LOYO folds with split-conformal intervals."""
    rows = []
    for fold in folds:
        train_base = fold.train_keys["base_onset"].to_numpy(dtype=float)
        y_raw = fold.y_train.to_numpy(dtype=float)
        usable = np.isfinite(train_base) & np.isfinite(y_raw)
        x_tr = fold.x_train.reset_index(drop=True).iloc[usable]
        y_tr_anom = y_raw[usable] - train_base[usable]
        years_tr = fold.train_keys["year"].to_numpy(dtype=int)[usable]

        # Inner 5-fold year-grouped OOF for conformal calibration
        uniq_y = sorted(set(years_tr.tolist()))
        grp = np.array([uniq_y.index(int(y)) % 5 for y in years_tr])
        oof = np.zeros(len(y_tr_anom))
        for k in range(5):
            held = grp == k
            oof[held] = shrink_factor * fit_predict_fn(x_tr.iloc[~held], y_tr_anom[~held], x_tr.iloc[held])
        res = y_tr_anom - oof
        q_low, q_high = float(np.quantile(res, 0.10)), float(np.quantile(res, 0.90))

        # Fit on full training years of this fold and predict held-out year
        x_te = fold.x_test.reset_index(drop=True)
        pred_anom = shrink_factor * fit_predict_fn(x_tr, y_tr_anom, x_te)
        base_te = fold.test_keys["base_onset"].to_numpy(dtype=float)
        clim_te = fold.test_keys["clim_onset"].to_numpy(dtype=float)
        y_te = fold.y_test.to_numpy(dtype=float)
        pred_doy = base_te + pred_anom

        for actual, p_doy, c_doy in zip(y_te, pred_doy, clim_te):
            if np.isfinite(actual) and np.isfinite(p_doy) and np.isfinite(c_doy):
                rows.append({
                    "year": fold.test_year,
                    "actual": actual,
                    "pred": p_doy,
                    "clim": c_doy,
                    "low": p_doy + q_low,
                    "high": p_doy + q_high,
                })
    df_res = pd.DataFrame(rows)
    mae_m = float(np.mean(np.abs(df_res["actual"] - df_res["pred"])))
    mae_c = float(np.mean(np.abs(df_res["actual"] - df_res["clim"])))
    rmse = float(np.sqrt(np.mean((df_res["actual"] - df_res["pred"]) ** 2)))
    cov = float(np.mean((df_res["actual"] >= df_res["low"]) & (df_res["actual"] <= df_res["high"])))
    skill = 1.0 - (mae_m / mae_c)
    return {"Model": name, "MAE (d)": round(mae_m, 3), "Clim MAE (d)": round(mae_c, 3), "Skill": round(skill, 4), "RMSE (d)": round(rmse, 3), "80% Coverage": round(cov, 3), "df": df_res}

# 1. Damped LightGBM (25 rounds, depth=2, leaves=3, L2=50)
def lgb_damped(x_tr, y_tr, x_te):
    import lightgbm as lgb
    params = {**M.BASE_PARAMS, "objective": "regression", "num_leaves": 3, "max_depth": 2, "lambda_l2": 50.0, "min_data_in_leaf": 120, "learning_rate": 0.02}
    ds = lgb.Dataset(x_tr, label=y_tr, free_raw_data=False)
    booster = lgb.train(params, ds, num_boost_round=25)
    return np.asarray(booster.predict(x_te), dtype=float)

# 2. Standardized Ridge Regression
def ridge_fn(x_tr, y_tr, x_te):
    pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=500.0))
    pipe.fit(x_tr, y_tr)
    return pipe.predict(x_te)

# 3. Standardized Huber Robust Regression (L1-like loss)
def huber_fn(x_tr, y_tr, x_te):
    pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), HuberRegressor(alpha=200.0, epsilon=1.2, max_iter=500))
    pipe.fit(x_tr, y_tr)
    return pipe.predict(x_te)

results = [
    {"Model": "1. Baseline LightGBM v3 (1-SE shrink)", "MAE (d)": bm["mae_model"], "Clim MAE (d)": bm["mae_clim"], "Skill": bm["skill"], "RMSE (d)": bm["rmse"], "80% Coverage": bm["coverage"]},
]
exp_runs = [
    evaluate_custom_loyo(folds, lgb_damped, "2. Damped LightGBM (depth=2, 25 rds, s=0.3)", shrink_factor=0.3),
    evaluate_custom_loyo(folds, ridge_fn, "3. Ridge L2 Anomaly (alpha=500, s=0.3)", shrink_factor=0.3),
    evaluate_custom_loyo(folds, huber_fn, "4. Huber Robust L1/L2 (alpha=200, s=0.3)", shrink_factor=0.3),
]
for r in exp_runs:
    results.append({k: v for k, v in r.items() if k != "df"})

comparison_df = pd.DataFrame(results)
display(comparison_df)

In [ ]:
# Visualize Actual vs Predicted Yearly Mean Onset across the 28 LOYO Folds
fig, ax = plt.subplots(figsize=(12, 4.5))
df_huber = exp_runs[2]["df"].groupby("year")[["actual", "pred", "clim", "low", "high"]].mean()

ax.plot(df_huber.index, df_huber["actual"], marker="o", color="#1d3557", linewidth=2, label="Observed National Mean Onset (DOY)")
ax.plot(df_huber.index, df_huber["clim"], linestyle="--", color="#6c757d", label="Fold Climatology Baseline")
ax.plot(df_huber.index, df_huber["pred"], marker="s", color="#e63946", linewidth=1.8, label="Huber Robust Anomaly Model (LOYO)")
ax.fill_between(df_huber.index, df_huber["low"], df_huber["high"], color="#e63946", alpha=0.15, label="80% Split-Conformal Interval")

ax.set_title("Out-of-Sample Leave-One-Year-Out Predictions vs Observed Monsoon Onset (1998–2025)")
ax.set_xlabel("Held-Out Test Year")
ax.set_ylabel("Monsoon Onset (DOY)")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

### Analysis of Model Comparison
- Comparing tree-based (`LightGBM`) and regularized linear/robust models (`Ridge`, `HuberRegressor`) on the exact same 28 LOYO folds shows why **conformal intervals** + **heavy shrinkage** are essential: year-to-year monsoon onset variability is dominated by May intraseasonal oscillations (MJO / BSISO) that occur *after* the April 25 issue date.
- Any model whose out-of-sample skill does not reliably beat `mae_clim` is safely shrunk to climatology so farmers are never given noisy point shifts, while still receiving a calibrated 80% conformal window (`~80.3%` empirical coverage).

## 5. Run Full Production Build & Export Artifacts (`metrics.json` & `onset_outlook.json`)

Running the cell below executes the complete `scripts/build_onset_ml.py` pipeline (both the `imerg` 1998–2025 primary arm and the `power_post2001` 2001–2025 secondary arm) and writes:
- `src/compute/ml/metrics.json`
- `web/public/data/onset_outlook.json`

In [ ]:
# Execute full production build script
!python scripts/build_onset_ml.py

# If running in Google Colab, offer one-click download of the generated JSON artifacts
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("src/compute/ml/metrics.json")
    files.download("web/public/data/onset_outlook.json")

## 6. Summary & Conclusions

1. **End-to-End Reproducibility:** The pipeline fetches 100% NASA/JAXA data directly from public AWS Open Data Zarr stores (`imerg_daily`, `merra2_daily`, `merra2_monthly`) with zero manual files or API keys.
2. **Leak-Free Temporal Validation:** All features and target baselines are constructed inside each Leave-One-Year-Out fold (`FoldClimatology`), ensuring zero information leakage across the 28 monsoon seasons.
3. **Shipped Out-of-Sample Skill & Calibrated Uncertainty:** The $L_1$ local-feature LightGBM model beats district climatology in **17/28 held-out seasons (`MAE 10.63 d` vs `10.73 d`, `skill = +0.009`, `status = "shipped"`)**, with 79.7% empirical coverage on its 80% split-conformal onset interval across 1,792 district-years.